Note: here are the links to see the results of each training session, conducted during this study of training yolov8s on custom dataset for gun detection:
- Training 1: https://app.clear.ml/projects/9aa9aa60ac6a4896ba8d1b6c137540d5/experiments/51f7987d37e54c58b1b117a11c90fa39/output/execution
- Training 2: https://app.clear.ml/projects/9aa9aa60ac6a4896ba8d1b6c137540d5/experiments/2fc6c6f30f184f6dbe0d2e71b3fba1b4/output/execution
- Training 3: https://app.clear.ml/projects/9aa9aa60ac6a4896ba8d1b6c137540d5/experiments/8272f5bc530a41e281a45e3e18447083/output/execution
- Training 4: https://app.clear.ml/projects/9aa9aa60ac6a4896ba8d1b6c137540d5/experiments/531bb5a9bff64562bd150e2124178d58/output/execution
- Training 5: https://app.clear.ml/projects/9aa9aa60ac6a4896ba8d1b6c137540d5/experiments/10940d6a01a34141abeb256c3f5c375a/output/execution

In [ ]:
# Setting our content directory as HOME
import os
HOME = os.getcwd()
print(HOME)

In [ ]:
# Pip install method, and importing ultralytics
!pip install ultralytics==8.0.20
from IPython import display
display.clear_output()
import ultralytics
ultralytics.checks()

In [ ]:
# Importing  libraries for weight and training and evaluation scripts provided by ultralytics
from ultralytics import YOLO
from IPython.display import display, Image

In [ ]:
# Downloading the dataset from Roboflow
!mkdir {HOME}/datasets
%cd {HOME}/datasets

In [ ]:
!pip install roboflow
from roboflow import Roboflow
import os
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("nizar-assad").project("pistols-lhjbh")
dataset = project.version(1645).download("yolov8")

## Custom Training

We will install and initialize ClearML for assessing the trainings

In [ ]:
!pip install clearml

In [ ]:
!clearml-init

In [ ]:
%env CLEARML_WEB_HOST=https://app.clear.ml
%env CLEARML_API_HOST=https://api.clear.ml
%env CLEARML_FILES_HOST=https://files.clear.ml
# Configure ClearML credentials through environment variables before running ClearML.\n# Never store ClearML secrets in a notebook.\n

# Training 1

In [ ]:
%cd {HOME}

!yolo task=detect mode=train model=yolov8s.pt data={dataset.location}/data.yaml epochs=25  lrf=0.001 lr0=0.001 batch=32 imgsz=800 plots=True

# Training 2

In [ ]:
%cd {HOME}

!yolo task=detect mode=train model=yolov8s.pt data={dataset.location}/data.yaml epochs=50  lrf=0.001 lr0=0.001 batch=32 imgsz=800 plots=True

# Training 3

In [ ]:
%cd {HOME}

!yolo task=detect mode=train model=yolov8s.pt data={dataset.location}/data.yaml epochs=25  lrf=0.01 lr0=0.01 batch=32 imgsz=800 plots=True

# Training 4

In [ ]:
%cd {HOME}

!yolo task=detect mode=train model=yolov8s.pt data={dataset.location}/data.yaml epochs=50  lrf=0.01 lr0=0.01 batch=16 imgsz=800 plots=True

# Training 5

In [ ]:
%cd {HOME}

!yolo task=detect mode=train model=yolov8s.pt data={dataset.location}/data.yaml epochs=50  lrf=0.001 lr0=0.001 batch=32 imgsz=800 plots=True

In [ ]:
# Vizualise the runs directory content
!ls {HOME}/runs/detect/train5/

In [ ]:
# confusion matrix for the last training
%cd {HOME}
Image(filename=f'{HOME}/runs/detect/train5/confusion_matrix.png', width=600)

In [ ]:
# Evaluation metrics (not needed as clearML provide more optmized vizualisations)
%cd {HOME}
Image(filename=f'{HOME}/runs/detect/train5/results.png', width=600)

In [ ]:
# Vizualise the validation batch (promising results)
%cd {HOME}
Image(filename=f'{HOME}/runs/detect/train4/val_batch0_pred.jpg', width=600)

## Validate Custom Model

In [ ]:
%cd {HOME}

!yolo task=detect mode=val model={HOME}/runs/detect/train4/weights/best.pt data={dataset.location}/data.yaml

In [ ]:
import shutil
import os
from google.colab import files

folder_path = '/content/runs/detect/train5/weights'
zip_file_path = '/content/weights2'

# Create a zip file for downloading the training content
shutil.make_archive(zip_file_path[:-4], 'zip', folder_path)

Inference with Custom Model yolov8s on unseen data using the best weight, after comparing the models and picking the best one.

In [ ]:
%cd {HOME}
!yolo task=detect mode=predict model={HOME}/best.pt conf=0.25 source=/content/test save=True

Now Let's take a look at few results, we will test our model on a unseen data to monitor the capacity of detecting in images that was not seen and unexpcted context.

In [ ]:
import glob
from IPython.display import Image, display

for image_path in glob.glob(f'{HOME}/runs/detect/predict/*.jpg'):
      display(Image(filename=image_path, width=600))
      print("\n")

In [ ]:
print("hello")

In [ ]:
from ultralytics import YOLO

model = YOLO("best.pt")  # load a pretrained model (recommended for training)
success = model.export(format="onnx")  # export the model to ONNX format